# Create test particle data (example)

This notebook shows an exemplary test particle calculation for generating the data required for the metadirect pair-correlation regularization.
Pregenerated data is provided in `matchinput.npy`.

In [ ]:
using Flux.Zygote, BSON, Flux, cuDNN, CUDA, Plots
include("utils.jl")

In [ ]:
BSON.@load "model_unregularized.bson" model
ϕ = zeros(150)
ϕ[1:100] .= 2 # choose pair potential
µ0 = 1
T = 1 # some functions are defined for T=1
L = 5
Gµ = Gµneural(µ0,ϕ,model);
V0(x) = 0
xs,ρ = minimize(L, μ0, T, ϕ, V0, model);

In [ ]:
c2b = get_c2_bulk(ρ[1],ϕ,model)
dx = 0.01
cϕ_tp = Gµ*(1/ρ[1] + sum(-c2b*dx));

In [ ]:
cϕ_AD = get_dc1dϕ_bulk(ρ[1],ϕ,model;dx = 0.01);

In [ ]:
plot(cϕ_AD, label = "autodiff", xlabel = "r/σ", ylabel = "χᵩᵇσ²")
plot!(cϕ_tp, label = "test particle") #assure that normalization is consistent in training.py 